In [1]:
import time
import pickle
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

print('Libraries loaded')

chrome_options = Options()
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')
chrome_options.add_argument('--disable-blink-features=AutomationControlled')

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=chrome_options
)

wait = WebDriverWait(driver, 20)

print('Driver ready!')

Libraries loaded
Driver ready!


In [2]:
MAX_PAGES = 30

# data variables
titles       = []
prices       = []
locations    = []
areas        = []
bedrooms     = []
bathrooms    = []
dates_posted = []
urls_list    = []

print('STEP 1: Scraping listing cards')

for page in range(1, MAX_PAGES + 1):

    url = f'https://www.olx.com.pk/property-for-rent_c3/?page={page}'
    print('Opening page', page)

    driver.get(url)

    wait.until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "li[aria-label='Listing']"))
    )

    driver.execute_script('window.scrollTo(0, document.body.scrollHeight)')
    time.sleep(2)

    cards = driver.find_elements(By.CSS_SELECTOR, "li[aria-label='Listing']")

    print('Cards found:', len(cards))

    for card in cards:

        try:

            title = card.find_element(
                By.CSS_SELECTOR, "div[aria-label='Title'] h2"
            ).text

            price = card.find_element(
                By.CSS_SELECTOR, "div[aria-label='Price'] span"
            ).text

            location = card.find_element(
                By.CSS_SELECTOR, "span[aria-label='Location']"
            ).text

            date = card.find_element(
                By.CSS_SELECTOR, "span[aria-label='Creation date']"
            ).text

            beds = card.find_element(
                By.CSS_SELECTOR, "span[aria-label='Beds']"
            ).text

            baths = card.find_element(
                By.CSS_SELECTOR, "span[aria-label='Bathrooms']"
            ).text

            area = card.find_element(
                By.CSS_SELECTOR, "span[aria-label='Area']"
            ).text

            link = card.find_element(
                By.CSS_SELECTOR, "a[href*='/item/']"
            ).get_attribute('href')

            titles.append(title)
            prices.append(price)
            locations.append(location)
            bedrooms.append(beds)
            bathrooms.append(baths)
            areas.append(area)
            dates_posted.append(date)
            urls_list.append(link)

        except:
            continue

    time.sleep(2)

print(f'\nStep 1 complete! Total listings collected: {len(urls_list)}')

STEP 1: Scraping listing cards
Opening page 1
Cards found: 48
Opening page 2
Cards found: 47
Opening page 3
Cards found: 48
Opening page 4
Cards found: 48
Opening page 5
Cards found: 47
Opening page 6
Cards found: 48
Opening page 7
Cards found: 48
Opening page 8
Cards found: 48
Opening page 9
Cards found: 48
Opening page 10
Cards found: 48
Opening page 11
Cards found: 47
Opening page 12
Cards found: 48
Opening page 13
Cards found: 48
Opening page 14
Cards found: 48
Opening page 15
Cards found: 48
Opening page 16
Cards found: 48
Opening page 17
Cards found: 48
Opening page 18
Cards found: 48
Opening page 19
Cards found: 48
Opening page 20
Cards found: 48
Opening page 21
Cards found: 48
Opening page 22
Cards found: 48
Opening page 23
Cards found: 48
Opening page 24
Cards found: 48
Opening page 25
Cards found: 48
Opening page 26
Cards found: 48
Opening page 27
Cards found: 48
Opening page 28
Cards found: 48
Opening page 29
Cards found: 48
Opening page 30
Cards found: 48

Step 1 complete! 

In [3]:
with open('urls_backup.pkl', 'wb') as f:
    pickle.dump(urls_list, f)

print(f'URLs backed up successfully! Total: {len(urls_list)}')

URLs backed up successfully! Total: 1040


In [4]:
descriptions  = []
seller_names  = []
image_counts  = []

print('STEP 2: Visiting listing pages')

def safe_text(soup, selector):
    el = soup.select_one(selector)
    if el:
        return el.get_text(' ', strip=True)
    return None

for i, url in enumerate(urls_list):
    try:
        driver.get(url)

        wait.until(
            EC.presence_of_element_located((By.TAG_NAME, 'body'))
        )

        soup = BeautifulSoup(driver.page_source, 'html.parser')

        # description
        description = safe_text(
            soup, "div[aria-label= 'Description']"
        )

        # seller name
        seller_label = soup.find("span", string="Posted by")
        seller = seller_label.find_next("span").get_text(strip=True) if seller_label else None

        # image count
        images = soup.select("div[data-aut-id='gallery'] img")
        img_count = len(images)

        descriptions.append(description)
        seller_names.append(seller)
        image_counts.append(img_count)

        print(f'[{i+1}/{len(urls_list)}] done')

        time.sleep(1)

    except:

        descriptions.append(None)
        seller_names.append(None)
        image_counts.append(None)

driver.quit()
print('\nStep 2 complete! Browser closed.')

STEP 2: Visiting listing pages
[1/1040] done
[2/1040] done
[3/1040] done
[4/1040] done
[5/1040] done
[6/1040] done
[7/1040] done
[8/1040] done
[9/1040] done
[10/1040] done
[11/1040] done
[12/1040] done
[13/1040] done
[14/1040] done
[15/1040] done
[16/1040] done
[17/1040] done
[18/1040] done
[19/1040] done
[20/1040] done
[21/1040] done
[22/1040] done
[23/1040] done
[24/1040] done
[25/1040] done
[26/1040] done
[27/1040] done
[28/1040] done
[29/1040] done
[30/1040] done
[31/1040] done
[32/1040] done
[33/1040] done
[34/1040] done
[35/1040] done
[36/1040] done
[37/1040] done
[38/1040] done
[39/1040] done
[40/1040] done
[41/1040] done
[42/1040] done
[43/1040] done
[44/1040] done
[45/1040] done
[46/1040] done
[47/1040] done
[48/1040] done
[49/1040] done
[50/1040] done
[51/1040] done
[52/1040] done
[53/1040] done
[54/1040] done
[55/1040] done
[56/1040] done
[57/1040] done
[58/1040] done
[59/1040] done
[60/1040] done
[61/1040] done
[62/1040] done
[63/1040] done
[64/1040] done
[65/1040] done
[66

In [5]:
# DATAFRAME CREATED
df = pd.DataFrame({
    'Title':         titles,
    'Price':         prices,
    'Location':      locations,
    'Area':          areas,
    'Bedrooms':      bedrooms,
    'Bathrooms':     bathrooms,
    'Date_Posted':   dates_posted,
    'Description':   descriptions,
    'Seller_Name':   seller_names,
    'Image_Count':   image_counts,
    'URL':           urls_list,
    'Label':         ''  
})

print('Total listings scraped:', len(df))
display(df)

df.to_csv('OLXdata_full1.csv', index=False, encoding='utf-8-sig')
print('OLXdata_full1.csv saved')

Total listings scraped: 1040


,Title,Price,Location,Area,Bedrooms,Bathrooms,Date_Posted,Description,Seller_Name,Image_Count,URL,Label
0,BRAND NEW CONDITION HOUSE TILES FLOORING & WOO...,Rs 2.25 Lac,"Johar Town, Lahore•",12 Marla,5 Beds,6 Baths,18 hours ago,Description NEAT AND CLEAN ENVIRONMENT \nTILES...,Malik Waseem,0,https://www.olx.com.pk/item/brand-new-conditio...,
1,3 Marla Brand New Spanish House Available For ...,"Rs 56,000","Al-Kabir Town - Phase 2, Lahore•",3 Marla,4 Beds,4 Baths,22 hours ago,Description 3m And 5m Brand New House Availabl...,kashif ali,0,https://www.olx.com.pk/item/3-marla-brand-new-...,
2,Villas available for rent in bahria town karachi,"Rs 28,000","Bahria Town - Precinct 31, Karachi•",235 SQYD,3 Beds,3 Baths,2 days ago,Description 24 hours electricity \n24 hours el...,waleed raza,0,https://www.olx.com.pk/item/villas-available-f...,
3,"Short-Time 1-Bedroom Flat for Couples Safe, Se...","Rs 3,000","Bahria Spring North, Rawalpindi•",570 SQFT,1 Bed,1 Bath,1 week ago,"Description *""Cozy 1-Bedroom Flat for Couples ...",Sardar Aqib,0,https://www.olx.com.pk/item/short-time-1-bedro...,
4,14 marla Ground Portion For Rent In Korang Town,"Rs 80,000","Korang Town, Islamabad•",14 Marla,3 Beds,4 Baths,41 minutes ago,Description 14 marla Ground Floor\n Having 3be...,bilal,0,https://www.olx.com.pk/item/14-marla-ground-po...,
...,...,...,...,...,...,...,...,...,...,...,...,...
1035,Semi Furnished 1-Bed Flat For Rent In Sector E...,"Rs 40,000","Bahria Town - Sector E, Lahore•",620 SQFT,1 Bed,1 Bath,11 hours ago,Description For More Details\nMahar Shahnawaz ...,Maher Shahnawaz,0,https://www.olx.com.pk/item/semi-furnished-1-b...,
1036,1 bed fully furnished flat for rent in Bahria ...,"Rs 43,000","Bahria Town - Sector C, Lahore•",520 SQFT,1 Bed,1 Bath,11 hours ago,Description For more information \nMahar Shahn...,Maher Shahnawaz,0,https://www.olx.com.pk/item/1-bed-fully-furnis...,
1037,1 Bed Fully Furnished Flat For Rent Hot Locati...,"Rs 46,000","Bahria Town - Sector E, Lahore•",560 SQFT,1 Bed,1 Bath,11 hours ago,Description Discover your dream home in a pres...,Maher Shahnawaz,0,https://www.olx.com.pk/item/1-bed-fully-furnis...,
1038,First Floor Available For Rent,"Rs 20,000","New Karachi - Sector 3, Karachi•",80 SQYD,2 Beds,2 Baths,11 hours ago,Description First Floor Available For Rent \n8...,Mehran Real Estate,0,https://www.olx.com.pk/item/first-floor-availa...,


OLXdata_full1.csv saved
